# Hypothesis Testing in Depth

Notebook for **Chapter 4** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch04.html

Tip: change a number, predict the output, then run the cell again.

## 4.1 The Logic of Testing, Seen Through Simulation

In [ ]:
import numpy as np
from scipy import stats
n, xbar, s, mu0 = 25, 480, 100, 500
t = (xbar - mu0) / (s / np.sqrt(n))
p = 2 * stats.t.sf(abs(t), n - 1)
print("t = %.3f  p = %.4f" % (t, p))
rng = np.random.default_rng(30)
sim = rng.normal(500, 100, (200_000, n))
tsim = (sim.mean(1) - mu0) / (sim.std(1, ddof=1) / np.sqrt(n))
print("simulated share with |t|>=1:", round((abs(tsim) >= 1).mean(), 4))

t = -1.000  p = 0.3273
simulated share with |t|>=1: 0.3276


In [ ]:
a = rng.normal(250, 60, (20_000, 40)); b = rng.normal(250, 60, (20_000, 40))
pv = stats.ttest_ind(a, b, axis=1).pvalue
print("share p<0.05: %.4f   p<0.10: %.4f   p<0.50: %.4f" % ((pv < .05).mean(), (pv < .10).mean(), (pv < .5).mean()))
print("counts per 0.1-wide bin:", np.histogram(pv, bins=10, range=(0, 1))[0])

share p<0.05: 0.0496   p<0.10: 0.0998   p<0.50: 0.5064
counts per 0.1-wide bin: [1997 2043 2016 2031 2041 1938 1966 2007 2055 1906]


## 4.2 z-Tests, t-Tests (One-sample, Two-sample, Paired) and the F-test

In [ ]:
res = stats.ttest_ind_from_stats(250, 60, 40, 225, 50, 45, equal_var=False)
se = np.sqrt(60**2 / 40 + 50**2 / 45)
dfw = (60**2/40 + 50**2/45)**2 / ((60**2/40)**2/39 + (50**2/45)**2/44)
ci = 25 + np.array([-1, 1]) * stats.t.ppf(0.975, dfw) * se
print("t = %.3f  p = %.4f  Welch df = %.1f  SE = %.2f" % (res.statistic, res.pvalue, dfw, se))
print("95%% CI for the difference: (%.1f, %.1f)" % tuple(ci))

t = 2.072  p = 0.0416  Welch df = 76.3  SE = 12.06
95% CI for the difference: (1.0, 49.0)


In [ ]:
rng = np.random.default_rng(31)
base = rng.uniform(100, 900, 12)
before = base + rng.normal(0, 10, 12)
after = base - 20 + rng.normal(0, 10, 12)
d = after - before
print("mean diff %.1f  sd of diff %.1f" % (d.mean(), d.std(ddof=1)))
print("paired   : t=%.2f p=%.2e" % tuple(stats.ttest_rel(after, before)))
print("unpaired : t=%.2f p=%.3f" % tuple(stats.ttest_ind(after, before, equal_var=False)))

mean diff -19.1  sd of diff 11.3
paired   : t=-5.86 p=1.09e-04
unpaired : t=-0.16 p=0.873


In [ ]:
x1, n1, x2, n2 = 500, 10_000, 600, 10_000
pp = (x1 + x2) / (n1 + n2)
z = (x2/n2 - x1/n1) / np.sqrt(pp * (1 - pp) * (1/n1 + 1/n2))
print("z = %.3f  p = %.4f  z^2 = %.2f" % (z, 2 * stats.norm.sf(z), z**2))
chi = stats.chi2_contingency([[x1, n1 - x1], [x2, n2 - x2]], correction=False)
print("chi-square = %.2f  p = %.4f" % (chi[0], chi[1]))
rng = np.random.default_rng(32)
g1 = rng.lognormal(5.3, 0.5, 200); g2 = rng.lognormal(5.3, 0.7, 200)
F = g1.var(ddof=1) / g2.var(ddof=1); pF = 2 * min(stats.f.cdf(F, 199, 199), stats.f.sf(F, 199, 199))
print("F test: F=%.3f p=%.2e   Levene: stat=%.2f p=%.2e" % ((F, pF) + tuple(stats.levene(g1, g2))))

z = 3.102  p = 0.0019  z^2 = 9.62
chi-square = 9.62  p = 0.0019
F test: F=0.472 p=1.66e-07   Levene: stat=13.05 p=3.42e-04


## 4.3 Chi-square Tests for Counts

In [ ]:
obs = np.array([280, 240, 260, 220])
r = stats.chisquare(obs, f_exp=[250] * 4)
print("chi2 = %.2f  p = %.4f  critical(5%%, 3 df) = %.3f" % (r.statistic, r.pvalue, stats.chi2.ppf(.95, 3)))
r10 = stats.chisquare(obs * 10, f_exp=[2500] * 4)
print("same shares, 10x data: chi2 = %.1f  p = %.2e" % (r10.statistic, r10.pvalue))

chi2 = 8.00  p = 0.0460  critical(5%, 3 df) = 7.815
same shares, 10x data: chi2 = 80.0  p = 3.07e-17


In [ ]:
tab = np.array([[300, 2700], [120, 1380], [60, 940]])
chi2, p, dof, exp = stats.chi2_contingency(tab)
V = np.sqrt(chi2 / (tab.sum() * min(tab.shape[0] - 1, tab.shape[1] - 1)))
print("expected counts:\n", np.round(exp, 1))
print("chi2 = %.2f  dof = %d  p = %.4f  Cramer's V = %.3f" % (chi2, dof, p, V))
print("conversion rates:", np.round(tab[:, 0] / tab.sum(1), 3))
print("Fisher exact p on tiny table [[3,1],[1,3]]: %.4f" % stats.fisher_exact([[3, 1], [1, 3]])[1])

expected counts:
 [[ 261.8 2738.2]
 [ 130.9 1369.1]
 [  87.3  912.7]]
chi2 = 16.43  dof = 2  p = 0.0003  Cramer's V = 0.055
conversion rates: [0.1  0.08 0.06]
Fisher exact p on tiny table [[3,1],[1,3]]: 0.4857


## 4.4 ANOVA and Post-hoc Tests

In [ ]:
A = [10, 12, 11, 13]; B = [14, 15, 13, 16]; C = [20, 19, 21, 20]
allv = np.concatenate([A, B, C]); gm = allv.mean()
ssb = sum(len(g) * (np.mean(g) - gm) ** 2 for g in (A, B, C))
ssw = sum(((np.array(g) - np.mean(g)) ** 2).sum() for g in (A, B, C))
F = (ssb / 2) / (ssw / 9)
print("SSB=%.2f SSW=%.2f SST=%.2f  F=%.2f  eta2=%.3f" % (ssb, ssw, ((allv - gm) ** 2).sum(), F, ssb / (ssb + ssw)))
print("scipy f_oneway:", stats.f_oneway(A, B, C))
print("Tukey HSD (group 0 = A, 1 = B, 2 = C):")
print(stats.tukey_hsd(A, B, C))

SSB=148.67 SSW=12.00 SST=160.67  F=55.75  eta2=0.925
scipy f_oneway: F_onewayResult(statistic=np.float64(55.74999999999986), pvalue=np.float64(8.504515585506145e-06))
Tukey HSD (group 0 = A, 1 = B, 2 = C):
Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -3.000     0.013    -5.280    -0.720
 (0 - 2)     -8.500     0.000   -10.780    -6.220
 (1 - 0)      3.000     0.013     0.720     5.280
 (1 - 2)     -5.500     0.000    -7.780    -3.220
 (2 - 0)      8.500     0.000     6.220    10.780
 (2 - 1)      5.500     0.000     3.220     7.780


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
rng = np.random.default_rng(33)
means = {"Van": [200, 190, 150], "Bike": [210, 205, 200]}
y = np.array([[rng.normal(m, 12, 10) for m in means[o]] for o in ("Van", "Bike")])   # shape (2, 3, 10)
a_, b_, n_ = y.shape
gm = y.mean(); mA = y.mean((1, 2)); mB = y.mean((0, 2)); mAB = y.mean(2)
ssA = b_ * n_ * ((mA - gm) ** 2).sum()
ssB = a_ * n_ * ((mB - gm) ** 2).sum()
ssAB = n_ * ((mAB - mA[:, None] - mB[None, :] + gm) ** 2).sum()
ssE = ((y - mAB[:, :, None]) ** 2).sum()
dfE = a_ * b_ * (n_ - 1)
rows = [("Vehicle", ssA, a_ - 1), ("Depot", ssB, b_ - 1), ("Vehicle x Depot", ssAB, (a_ - 1) * (b_ - 1))]
print(f"{'source':12}{'SS':>10}{'df':>4}{'F':>9}{'p':>12}")
for name, ss, df in rows:
    Fv = (ss / df) / (ssE / dfE)
    print(f"{name:12}{ss:10.1f}{df:4d}{Fv:9.2f}{stats.f.sf(Fv, df, dfE):12.2e}")
print(f"{'Error':12}{ssE:10.1f}{dfE:4d}")
print("sum of SS minus SS_total:", abs(round(ssA + ssB + ssAB + ssE - ((y - gm) ** 2).sum(), 9)))
print("cell means:\n", np.round(mAB, 1))

source              SS  df        F           p
Vehicle        12628.6   1   104.89    2.92e-14
Depot          10740.0   2    44.60    3.66e-12
Vehicle x Depot    6113.2   2    25.39    1.69e-08
Error           6501.3  54
sum of SS minus SS_total: 0.0
cell means:
 [[203.  186.7 147.5]
 [214.4 205.1 204.8]]


## 4.5 Non-parametric Tests, Permutation Tests and Normality Tests

In [ ]:
rng = np.random.default_rng(34)
A = rng.lognormal(np.log(200), 0.6, 30)
B = rng.lognormal(np.log(160), 0.6, 30)
print("medians:", round(np.median(A), 1), round(np.median(B), 1), " means:", round(A.mean(), 1), round(B.mean(), 1))
print("Welch t        : p = %.4f" % stats.ttest_ind(A, B, equal_var=False).pvalue)
print("Mann-Whitney U : p = %.4f" % stats.mannwhitneyu(A, B, alternative="two-sided").pvalue)
obs = A.mean() - B.mean(); pool = np.concatenate([A, B]); cnt = 0
for _ in range(10_000):
    rng.shuffle(pool)
    cnt += abs(pool[:30].mean() - pool[30:].mean()) >= abs(obs)
print("permutation (manual): p = %.4f" % ((cnt + 1) / 10_001))
pt = stats.permutation_test((A, B), lambda x, y: x.mean() - y.mean(), n_resamples=9999, random_state=1)
print("permutation (scipy)  : p = %.4f" % pt.pvalue)
before = rng.lognormal(np.log(200), 0.5, 15); after = before * rng.lognormal(np.log(0.85), 0.1, 15)
print("Wilcoxon signed-rank on 15 pairs: p = %.2e   paired t: p = %.2e" % (stats.wilcoxon(before, after).pvalue, stats.ttest_rel(before, after).pvalue))

medians: 221.3 136.5  means: 244.3 172.3
Welch t        : p = 0.0351
Mann-Whitney U : p = 0.0067
permutation (manual): p = 0.0246
permutation (scipy)  : p = 0.0196
Wilcoxon signed-rank on 15 pairs: p = 6.10e-05   paired t: p = 4.45e-06


In [ ]:
rng = np.random.default_rng(35)
g = [rng.lognormal(np.log(m), 0.6, 25) for m in (180, 200, 260)]
print("ANOVA p = %.4f   Kruskal-Wallis p = %.4f" % (stats.f_oneway(*g).pvalue, stats.kruskal(*g).pvalue))
allx = np.concatenate(g)
print("Shapiro raw  : W=%.3f p=%.2e" % stats.shapiro(allx))
print("Shapiro log  : W=%.3f p=%.3f" % stats.shapiro(np.log(allx)))
print("KS (params estimated) on log data: p=%.3f" % stats.kstest((np.log(allx) - np.log(allx).mean()) / np.log(allx).std(ddof=1), "norm").pvalue)
big = stats.t(10).rvs(5000, random_state=7)
print("Shapiro on 5000 t(10) values (almost normal): p=%.4f" % stats.shapiro(big).pvalue)

ANOVA p = 0.1544   Kruskal-Wallis p = 0.2414
Shapiro raw  : W=0.889 p=7.66e-06
Shapiro log  : W=0.981 p=0.309
KS (params estimated) on log data: p=0.764
Shapiro on 5000 t(10) values (almost normal): p=0.0000


## 4.6 Power and Sample Size

In [ ]:
za, zb = stats.norm.ppf(0.975), stats.norm.ppf(0.80)
n_req = 2 * (za + zb) ** 2 * 100**2 / 20**2
print("required n per group: %.1f  (z_alpha/2=%.3f, z_beta=%.3f)" % (n_req, za, zb))
rng = np.random.default_rng(36)
def power(n, delta=20, sd=100, reps=5000):
    a = rng.normal(0, sd, (reps, n)); b = rng.normal(delta, sd, (reps, n))
    return (stats.ttest_ind(a, b, axis=1).pvalue < 0.05).mean()
for n in (50, 100, 200, 393, 600):
    print("n=%-4d simulated power = %.3f" % (n, power(n)))

required n per group: 392.4  (z_alpha/2=1.960, z_beta=0.842)
n=50   simulated power = 0.167
n=100  simulated power = 0.293
n=200  simulated power = 0.504
n=393  simulated power = 0.805
n=600  simulated power = 0.931


In [ ]:
def n_prop(p1, p2, alpha=0.05, power=0.8):
    za, zb = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    pb = (p1 + p2) / 2
    return (za * np.sqrt(2 * pb * (1 - pb)) + zb * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2 / (p1 - p2) ** 2
for p2 in (0.06, 0.055):
    print("5%% -> %.1f%%: n per arm = %.0f" % (p2 * 100, n_prop(0.05, p2)))
nn = int(np.ceil(n_prop(0.05, 0.06)))
rng = np.random.default_rng(37); rej = 0; reps = 4000
for _ in range(reps):
    x1, x2 = rng.binomial(nn, 0.05), rng.binomial(nn, 0.06)
    pp = (x1 + x2) / (2 * nn)
    z = (x2 - x1) / nn / np.sqrt(pp * (1 - pp) * 2 / nn)
    rej += 2 * stats.norm.sf(abs(z)) < 0.05
print("simulated power at n=%d: %.3f" % (nn, rej / reps))

5% -> 6.0%: n per arm = 8158
5% -> 5.5%: n per arm = 31233
simulated power at n=8158: 0.816


In [ ]:
# plot
import matplotlib.pyplot as plt
ns = np.array([20, 50, 100, 200, 300, 393, 500, 700, 1000])
plt.figure(figsize=(6, 3.6))
for d_, c_ in [(10, "#2ca02c"), (20, "#1f77b4"), (30, "#d62728")]:
    plt.plot(ns, [power(int(k), delta=d_, reps=2000) for k in ns], marker="o", color=c_, label=f"true gain {d_} min")
plt.axhline(0.8, color="gray", ls="--"); plt.xlabel("parcels per depot"); plt.ylabel("power"); plt.legend(); plt.title("Power curves (SD = 100 min, alpha = 0.05)")
plt.tight_layout(); plt.savefig(f"{IMG}/ch04_power.png", dpi=120); plt.close()

## Chapter recap

## Practice set

Problem 4.7.1 A test statistic is z = 2.58 (two-sided). Is the p-value below 0.01?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch04.html